---슈도 코드
모델 지정  
Langchain으로 크로마db 연결  
질문이 들어오면 RAG의 벡터 DB에서 유사도 검색으로 답변  
답변할때 RAG의 근거 데이터 row를 반환  

시간이 남으면 질문 입력 받을때 강아지의 나이를 넣을 내용  
생후진료처 기준 아기견(~2), 성견(2~6), 노령견(7~)으로 바뀌게  
검색 정확도를 높임  

In [5]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI


# 노트북 실행 위치가 프로젝트 루트이거나 notebooks 폴더인 경우 모두 지원합니다.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").is_dir() and (PROJECT_DIR.parent / "data").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

load_dotenv(PROJECT_DIR / ".env")
CHROMA_DIR = PROJECT_DIR / "data" / "chroma_db"
TRAINING_CSV = PROJECT_DIR / "data" / "training" / "df.csv"

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    encode_kwargs={"normalize_embeddings": True},
)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0) if os.getenv("OPENAI_API_KEY") else None
parser = StrOutputParser()

print(f"프로젝트 경로: {PROJECT_DIR}")
print(f"ChromaDB 경로: {CHROMA_DIR}")
if model is None:
    print("OPENAI_API_KEY가 없어 검색만 준비했습니다. 답변 생성 전 .env 또는 환경 변수에 키를 설정하세요.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4951.37it/s]


프로젝트 경로: c:\Users\rhksa\OneDrive\사진\바탕 화면\mle-01-p1-team2
ChromaDB 경로: c:\Users\rhksa\OneDrive\사진\바탕 화면\mle-01-p1-team2\data\chroma_db
OPENAI_API_KEY가 없어 검색만 준비했습니다. 답변 생성 전 .env 또는 환경 변수에 키를 설정하세요.


In [7]:
vector_db = Chroma(
    collection_name="pet_care",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

INDEX_LIMIT = 100
if vector_db._collection.count() == 0:
    df = pd.read_csv(TRAINING_CSV)
    df = df.dropna(subset=["qa.input"]).head(INDEX_LIMIT).reset_index(drop=True)
    documents = df["qa.input"].astype(str).tolist()
    ids = (
        df["Unnamed: 0"].astype(str).tolist()
        if "Unnamed: 0" in df.columns
        else [str(index) for index in df.index]
    )
    metadatas = [
        {
            key: "" if pd.isna(value) else str(value)
            for key, value in row.items()
            if key != "qa.input"
        }
        for _, row in df.iterrows()
    ]
    vector_db.add_texts(texts=documents, metadatas=metadatas, ids=ids)
    print(f"training/df.csv에서 {len(documents):,}개 문서를 새로 적재했습니다.")

print(f"Chroma 컬렉션 문서 수: {vector_db._collection.count():,}")


training/df.csv에서 100개 문서를 새로 적재했습니다.
Chroma 컬렉션 문서 수: 100


In [3]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """아래 [검색 데이터]를 근거로 사용자의 질문에 답하세요.
규칙:
1. 검색된 데이터에 근거해서만 답변하세요.
2. 데이터에 없는 내용은 임의로 추측하지 마세요.
3. 답변은 간결하게 작성하세요.

[검색 데이터]
{context}
""",
    ),
    ("human", "{question}"),
])
rag_chain = prompt | model | parser if model is not None else None


In [8]:
def ask_rag(question, k=3):
    docs = vector_db.similarity_search(question, k=k)
    context = "\n\n".join(
        f"질문: {doc.page_content}\n답변: {doc.metadata.get('qa.output', '')}"
        for doc in docs
    )

    if rag_chain is None:
        answer = "유사도 검색은 성공했습니다. 답변 생성에는 OPENAI_API_KEY가 필요합니다."
    else:
        answer = rag_chain.invoke({"context": context, "question": question})

    evidence_rows = [doc.metadata for doc in docs]
    return {"answer": answer, "evidence_rows": evidence_rows}


In [9]:
result = ask_rag("여기에 질문 입력") 
print("답변:") 
print(result["answer"]) 

print("\n근거 데이터 row:") 
for row in result["evidence_rows"]: 
    print(row)

답변:
유사도 검색은 성공했습니다. 답변 생성에는 OPENAI_API_KEY가 필요합니다.

근거 데이터 row:
{'meta.lifeCycle': '노령견', 'meta.disease': '기타', 'qa.output': '말씀주신 사항만으로는 정확한 판단이 어렵지만 아마도 강아지가 구토한 것으로 판단됩니다. 또한 보호자가 보셨다는 실 같은 것은 제가 확인이 어렵기에 말씀드리기도 역시 어렵습니다. 하지만 기생충의 일종일 수도 있으니 이와 관련하여 추가적으로 관찰하고 주의 깊게 지켜보시는 것이 중요합니다. 그리고 정확한 진단을 위해 즉시 동물병원에 방문하여 검사를 받아야 하며, 토사물 사진이나 샘플을 가져가면 진단에 도움이 됩니다.', 'qa.instruction': '너는 반려견 진료 경험이 많은 수의사야. 보호자의 질문에 답해줘.', 'Unnamed: 0': '83', 'meta.department': '내과'}
{'qa.instruction': '너는 반려견의 건강 문제를 조기에 발견할 수 있는 전문가야. 보호자의 질문에 대해 답해줘.', 'meta.department': '내과', 'meta.lifeCycle': '자견', 'meta.disease': '기타', 'qa.output': '마가 크로칩 등록 후 2~3일 간 기다려야 하는 이유는, 등록된 정보가 시스템에 반영되는 데 일정 시간이 소요되기 때문입니다. 이 기간 동안 강아지에게 발생할 수 있는 것으로 문제는 감염의 위험이 있는 것으로 데, 목욕을 시키면 주사 부위에 물이 들어가 감염을 유발할 수 있기에 주의가 필요합니다. 강아지가 많이 더러워져서 목욕시키고 싶으신 마음은 이해하지만, 현재로서는 리스크가 높습니다. 따라서 목욕 대신 강아지의 발이 나 더러워진 부분만 물티슈를 이용해 부드럽게 닦아주는 것이 좋습니다. 개선되면 필요시 유지해 주세요 불편해하면 강도는 낮춰 주세요. 주 1~2회 경과를 보며 강도나 빈도를 조정해 주세요 경과 사진을 주 1~2회 정도 남겨 주세요. 환경은 자극원을 최소화하